In [3]:
import os
import ssl
import certifi

# Disable SSL verification completely for requests library
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['SSL_CERT_FILE'] = ''

ssl._create_default_https_context = ssl._create_unverified_context

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold, LeaveOneOut
from sklearn.linear_model    import LinearRegression
from sklearn.preprocessing   import OrdinalEncoder, OneHotEncoder  
from sklearn.impute          import SimpleImputer
from sklearn.metrics         import mean_squared_error, r2_score
from tqdm                    import tqdm

random_seed = 42

In [4]:
# Download the latest version of the dataset
path = kagglehub.dataset_download("shashanknecrothapa/ames-housing-dataset")

print("Path to dataset files:", path)

# Construct the full path to the CSV file (update the file name if necessary)
csv_file = os.path.join(path, "AmesHousing.csv")

# Read the dataset into a DataFrame
df = pd.read_csv(csv_file)

Path to dataset files: /Users/gkumargaur/.cache/kagglehub/datasets/shashanknecrothapa/ames-housing-dataset/versions/1


In [5]:
df_clean = df.drop(columns=['Order','PID'])    # makes a copy
df_clean.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [6]:
# Although the info method above shows us which features have missing values and how many aren't missing,
# this function will provide more details about the missing values. 

def show_null_counts_features(df):
    # Count the nulls and calculate the %
    count_nulls = df.isnull().sum() 
    df_nulls = (df.isnull().mean() * 100).round(2)
    
    # Determine if the column is numeric or non-numeric
    feature_types = df.dtypes.apply(lambda x: 'Numeric' if np.issubdtype(x, np.number) else 'Categorical')
    
    # Filter out the columns with missing values and sort them in descending order
    missing_data = pd.DataFrame({
        'Feature': count_nulls[count_nulls > 0].index,
        '# Null Values': count_nulls[count_nulls > 0].values, 
        'Null %': df_nulls[df_nulls > 0].values,
        'Type': feature_types[count_nulls > 0].values
    }).sort_values(by='Null %', ascending=False)
    
    print(f'The dataset contains {len(df)} samples.\n')

    if (len(missing_data) == 0):
        print("There are no null values in the dataset!")
    else:
        # Print null value stats
        print('Feature Name    # Nulls      Null %    Type')
        print('------------    -------      ------    ----')
        for index, row in missing_data.iterrows():
            print(f"{row['Feature']:<15} {row['# Null Values']:<12} {row['Null %']:.2f}%   {row['Type']}")


# Uncomment to see results
            
show_null_counts_features(df_clean)

The dataset contains 2930 samples.

Feature Name    # Nulls      Null %    Type
------------    -------      ------    ----
Pool QC         2917         99.56%   Categorical
Misc Feature    2824         96.38%   Categorical
Alley           2732         93.24%   Categorical
Fence           2358         80.48%   Categorical
Mas Vnr Type    1775         60.58%   Categorical
Fireplace Qu    1422         48.53%   Categorical
Lot Frontage    490          16.72%   Numeric
Garage Cond     159          5.43%   Categorical
Garage Qual     159          5.43%   Categorical
Garage Finish   159          5.43%   Categorical
Garage Yr Blt   159          5.43%   Numeric
Garage Type     157          5.36%   Categorical
Bsmt Exposure   83           2.83%   Categorical
BsmtFin Type 2  81           2.76%   Categorical
Bsmt Cond       80           2.73%   Categorical
Bsmt Qual       80           2.73%   Categorical
BsmtFin Type 1  80           2.73%   Categorical
Mas Vnr Area    23           0.78%   Numeric

- count_nulls = df.isnull().sum() 
    So count_nulls gives you null counts per column, which is useful for identifying which features have missing data.

- To get the TOTAL null count across the entire DataFrame:
    total_nulls = df.isnull().sum().sum()  # Double sum

- To see which columns have nulls:
    columns_with_nulls = count_nulls[count_nulls > 0]

        



In [7]:
max_nulls = 500      # we will drop any features with more than max_nulls missing values

# Count null values per column
count_nulls = df_clean.isnull().sum()

# to check
print(count_nulls[count_nulls > max_nulls])

# Filter out columns where null count exceeds max_nulls
columns_to_drop = count_nulls[count_nulls > max_nulls].index.tolist()

# Drop the columns
df_clean = df_clean.drop(columns=columns_to_drop)

# Uncomment to verify they were removed
df_clean

# show_null_counts_features(df_clean)
show_null_counts_features(df_clean)

Alley           2732
Mas Vnr Type    1775
Fireplace Qu    1422
Pool QC         2917
Fence           2358
Misc Feature    2824
dtype: int64
The dataset contains 2930 samples.

Feature Name    # Nulls      Null %    Type
------------    -------      ------    ----
Lot Frontage    490          16.72%   Numeric
Garage Yr Blt   159          5.43%   Numeric
Garage Qual     159          5.43%   Categorical
Garage Finish   159          5.43%   Categorical
Garage Cond     159          5.43%   Categorical
Garage Type     157          5.36%   Categorical
Bsmt Exposure   83           2.83%   Categorical
BsmtFin Type 2  81           2.76%   Categorical
BsmtFin Type 1  80           2.73%   Categorical
Bsmt Cond       80           2.73%   Categorical
Bsmt Qual       80           2.73%   Categorical
Mas Vnr Area    23           0.78%   Numeric
Bsmt Full Bath  2            0.07%   Numeric
Bsmt Half Bath  2            0.07%   Numeric
BsmtFin SF 1    1            0.03%   Numeric
BsmtFin SF 2    1        

In [8]:
# Identify categorical and numeric features

categorical_features = df_clean.select_dtypes(exclude=['number']).columns.tolist()
numeric_features     = df_clean.select_dtypes(include=['number']).columns.tolist()

# Uncomment to print results

print("Categorical Features:", categorical_features)
print()
print("Numeric Features:", numeric_features)

Categorical Features: ['MS Zoning', 'Street', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC', 'Central Air', 'Electrical', 'Kitchen Qual', 'Functional', 'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond', 'Paved Drive', 'Sale Type', 'Sale Condition']

Numeric Features: ['MS SubClass', 'Lot Frontage', 'Lot Area', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd', 'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Ar

In [9]:
# First make a copy of the cleaned dataset
df_imputed = df_clean.copy()

# Impute categorical columns (using most frequent category)
imputer = SimpleImputer(strategy='most_frequent')
df_imputed[categorical_features] = imputer.fit_transform(df_imputed[categorical_features])

# Verify: only numeric features should appear
imputed_count_nulls = df_imputed.isnull().sum()
# See the data type of each column
print(imputed_count_nulls.dtypes)





int64


In [13]:
# Count nulls per column after imputation
nulls_after_imputation = df_imputed.isnull().sum()

# Get columns that still have nulls
cols_with_nulls = nulls_after_imputation[nulls_after_imputation > 0]

if len(cols_with_nulls) > 0:
    print("Columns with remaining nulls:")
    print(cols_with_nulls)
    
    # Check if these columns are numeric
    for col in cols_with_nulls.index:
        col_type = df_imputed[col].dtype
        is_numeric = np.issubdtype(col_type, np.number)
        print(f"{col}: {col_type} - Numeric: {is_numeric}")
else:
    print("✓ No null values remaining - imputation successful!")

Columns with remaining nulls:
Lot Frontage      490
Mas Vnr Area       23
BsmtFin SF 1        1
BsmtFin SF 2        1
Bsmt Unf SF         1
Total Bsmt SF       1
Bsmt Full Bath      2
Bsmt Half Bath      2
Garage Yr Blt     159
Garage Cars         1
Garage Area         1
dtype: int64
Lot Frontage: float64 - Numeric: True
Mas Vnr Area: float64 - Numeric: True
BsmtFin SF 1: float64 - Numeric: True
BsmtFin SF 2: float64 - Numeric: True
Bsmt Unf SF: float64 - Numeric: True
Total Bsmt SF: float64 - Numeric: True
Bsmt Full Bath: float64 - Numeric: True
Bsmt Half Bath: float64 - Numeric: True
Garage Yr Blt: float64 - Numeric: True
Garage Cars: float64 - Numeric: True
Garage Area: float64 - Numeric: True


In [14]:
# Get columns with nulls
cols_with_nulls = df_imputed.columns[df_imputed.isnull().any()].tolist()

# Get non-numeric columns
non_numeric_cols = df_imputed.select_dtypes(exclude=[np.number]).columns.tolist()

# Check if any non-numeric columns still have nulls
non_numeric_with_nulls = [col for col in cols_with_nulls if col in non_numeric_cols]

if len(non_numeric_with_nulls) > 0:
    print(f"❌ Problem! Non-numeric columns still have nulls: {non_numeric_with_nulls}")
else:
    print("✓ All non-numeric columns have been imputed (no nulls in categorical columns)")
    
if len(cols_with_nulls) > 0:
    print(f"Columns with remaining nulls (should be numeric only): {cols_with_nulls}")
else:
    print("✓ No null values in any column!")

✓ All non-numeric columns have been imputed (no nulls in categorical columns)
Columns with remaining nulls (should be numeric only): ['Lot Frontage', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Yr Blt', 'Garage Cars', 'Garage Area']


In [15]:
# First make a copy of the cleaned dataset
df_imputed = df_clean.copy()

# Impute categorical columns (using most frequent category)
imputer = SimpleImputer(strategy='most_frequent')
df_imputed[categorical_features] = imputer.fit_transform(df_imputed[categorical_features])

# Verify: only numeric features should appear
nulls_after_imputation = df_imputed.isnull().sum()
# Get columns that still have nulls
cols_with_nulls = nulls_after_imputation[nulls_after_imputation > 0]
# Check if these columns are numeric
for col in cols_with_nulls.index:
    col_type = df_imputed[col].dtype
    is_numeric = np.issubdtype(col_type, np.number)
    print(f"{col}: {col_type} - Numeric: {is_numeric}")

show_null_counts_features(df_imputed)

Lot Frontage: float64 - Numeric: True
Mas Vnr Area: float64 - Numeric: True
BsmtFin SF 1: float64 - Numeric: True
BsmtFin SF 2: float64 - Numeric: True
Bsmt Unf SF: float64 - Numeric: True
Total Bsmt SF: float64 - Numeric: True
Bsmt Full Bath: float64 - Numeric: True
Bsmt Half Bath: float64 - Numeric: True
Garage Yr Blt: float64 - Numeric: True
Garage Cars: float64 - Numeric: True
Garage Area: float64 - Numeric: True
The dataset contains 2930 samples.

Feature Name    # Nulls      Null %    Type
------------    -------      ------    ----
Lot Frontage    490          16.72%   Numeric
Garage Yr Blt   159          5.43%   Numeric
Mas Vnr Area    23           0.78%   Numeric
Bsmt Full Bath  2            0.07%   Numeric
Bsmt Half Bath  2            0.07%   Numeric
BsmtFin SF 1    1            0.03%   Numeric
BsmtFin SF 2    1            0.03%   Numeric
Bsmt Unf SF     1            0.03%   Numeric
Total Bsmt SF   1            0.03%   Numeric
Garage Cars     1            0.03%   Numeric
Gara

In [16]:
df_imputed['Garage Qual'].value_counts(dropna=False)

Garage Qual
TA    2774
Fa     124
Gd      24
Po       5
Ex       3
Name: count, dtype: int64

In [17]:
# Impute numeric columns (using the median)
numeric_imputer = SimpleImputer(strategy='median')
df_imputed[numeric_features] = numeric_imputer.fit_transform(df_imputed[numeric_features])

# Verify: There should be no null values
nulls_after_imputation = df_imputed.isnull().sum()
# Get columns that still have nulls
cols_with_nulls = nulls_after_imputation[nulls_after_imputation > 0]
# Check if these columns are numeric
for col in cols_with_nulls.index:
    col_type = df_imputed[col].dtype
    is_numeric = np.issubdtype(col_type, np.number)
    print(f"{col}: {col_type} - Numeric: {is_numeric}")

show_null_counts_features(df_imputed)

The dataset contains 2930 samples.

There are no null values in the dataset!


-- The question asks for "the number of ROWS with missing values", but you're providing the number of COLUMNS with missing values.
Your current answer (len(cols_with_nulls.index)) gives the count of columns, not rows.

In [19]:
# Option 1: Count rows with ANY null value
a1c = df_imputed.isnull().any(axis=1).sum()
print(a1c)


0


In [20]:
# Put df_imputed in the form X, y
X = df_imputed.drop(columns=['SalePrice'])
y = df_imputed['SalePrice']

# Initialize OrdinalEncoder
ordinal_encoder = OrdinalEncoder()

# Convert categorical features to ordinal encoding
X[categorical_features] = ordinal_encoder.fit_transform(X[categorical_features])

# Convert back to DataFrame to retain column names 
df_ordinal_encoded = pd.DataFrame(X, columns=X.columns)


# # Verify: all Dtypes should be float64
for col in df_ordinal_encoded[categorical_features].columns:
    print(f"{col}: {df_ordinal_encoded[col].dtype}")

# X.info()

MS Zoning: float64
Street: float64
Lot Shape: float64
Land Contour: float64
Utilities: float64
Lot Config: float64
Land Slope: float64
Neighborhood: float64
Condition 1: float64
Condition 2: float64
Bldg Type: float64
House Style: float64
Roof Style: float64
Roof Matl: float64
Exterior 1st: float64
Exterior 2nd: float64
Exter Qual: float64
Exter Cond: float64
Foundation: float64
Bsmt Qual: float64
Bsmt Cond: float64
Bsmt Exposure: float64
BsmtFin Type 1: float64
BsmtFin Type 2: float64
Heating: float64
Heating QC: float64
Central Air: float64
Electrical: float64
Kitchen Qual: float64
Functional: float64
Garage Type: float64
Garage Finish: float64
Garage Qual: float64
Garage Cond: float64
Paved Drive: float64
Sale Type: float64
Sale Condition: float64


In [21]:
# Your answer here; use an expression, not a constant derived by examining the data

df_ordinal_encoded['Lot Shape'].value_counts(dropna=False)                        # replace 0 with an expression

Lot Shape
3.0    1859
0.0     979
1.0      76
2.0      16
Name: count, dtype: int64

In [22]:
# Your code here, add additional code cells if you wish

# Step 1: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)

# Step 2: Create a linear model and fit it to the training data
linearmodel = LinearRegression()


# Step 3: Perform K-Fold Cross-Validation with K = 5
scores = cross_val_score(linearmodel, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')

# Step 4: Calculate the mean of the CV scores and negate the result
cv_rmse = -scores.mean()
print('negated mean_squared_error of RMSE measurements over all K folds', cv_rmse)

# Step 5: now fit the model to the training data
linearmodel.fit(X_train, y_train)

# Step 6: Calculate the Test RMSE
test_rmse = np.sqrt(mean_squared_error(y_test, linearmodel.predict(X_test)))
print('Test RMSE', test_rmse)




negated mean_squared_error of RMSE measurements over all K folds 31176.271420748195
Test RMSE 33445.54872714283


In [23]:
# Your code here; add additional cells if you wish
rkf = RepeatedKFold(n_splits=5, n_repeats=100, random_state=random_seed)

# Step 1: Perform Repeated K-Fold Cross-Validation with K = 5 and n_repeats=100
scores = cross_val_score(linearmodel, X_train, y_train, cv=rkf, scoring='neg_root_mean_squared_error')

# Step 2: Calculate the mean of the CV scores and negate the result
cv_rmse = -scores.mean()
print('negated mean_squared_error of RMSE measurements over all 100*K folds', cv_rmse)

# Step 3: Fit the model to the training data
linearmodel.fit(X_train, y_train)

# Step 4: Calculate the Test RMSE
test_rmse = np.sqrt(mean_squared_error(y_test, linearmodel.predict(X_test)))
print('Test RMSE', test_rmse)


negated mean_squared_error of RMSE measurements over all 100*K folds 32189.411453329285
Test RMSE 33445.54872714283


In [24]:
# Your code here; add additional cells if you wish
loo = LeaveOneOut()

# Step 1: Perform Leave-One-Out Cross-Validation
scores = cross_val_score(linearmodel, X_train, y_train, cv=loo, scoring='neg_root_mean_squared_error')

# Step 2: Calculate the mean of the CV scores and negate the result 
cv_rmse = -scores.mean()
print('negated mean_squared_error of RMSE measurements over all K folds', cv_rmse)

# Step 3: Fit the model to the training data
linearmodel.fit(X_train, y_train)

# Step 4: Calculate the Test RMSE
test_rmse = np.sqrt(mean_squared_error(y_test, linearmodel.predict(X_test)))
print('Test RMSE', test_rmse)

negated mean_squared_error of RMSE measurements over all K folds 19107.22548426038
Test RMSE 33445.54872714283


In [25]:
# Your code here; add additional cells if you wish
import time
from sklearn.model_selection import cross_val_predict
loo = LeaveOneOut()
linearmodel_loo = LinearRegression()

# Step 1: Perform Leave-One-Out Cross-Validation
# scores_loo = cross_val_score(linearmodel_loo, X_train, y_train, cv=loo, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
# Use cross_val_predict to get predictions for all samples
y_pred_loo = cross_val_predict(linearmodel_loo, X_train, y_train, cv=loo, n_jobs=-1, verbose=1)

# Add small delay to let verbose output finish
time.sleep(0.5)
# Step 2: Calculate the mean of the CV scores and negate the result 
# cv_rmse_loo = -scores_loo.mean()
# Calculate RMSE correctly: sqrt(mean(squared errors))
cv_rmse_loo = np.sqrt(mean_squared_error(y_train, y_pred_loo))
print('negated mean_squared_error of RMSE measurements over all K folds', cv_rmse_loo)

# Step 3: Fit the model to the training data
linearmodel_loo.fit(X_train, y_train)

# Step 4: Calculate the Test RMSE
test_rmse_loo = np.sqrt(mean_squared_error(y_test, linearmodel_loo.predict(X_test)))
print('Test RMSE', test_rmse_loo)



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:    4.6s
[Parallel(n_jobs=-1)]: Done 820 tasks      | elapsed:    6.1s
[Parallel(n_jobs=-1)]: Done 2344 out of 2344 | elapsed:    8.6s finished


negated mean_squared_error of RMSE measurements over all K folds 32458.21510199348
Test RMSE 33445.54872714283


### Old approach (WRONG):
Uses cross_val_score with neg_root_mean_squared_error
Computes RMSE for each individual sample
Takes mean of 2,344 individual RMSEs
Result: 19,107 (too low!)
### New approach (CORRECT):
Uses cross_val_predict to get predictions for all samples
Computes RMSE once over all predictions together
Result: Should be ~31,000-32,000 (closer to test RMSE)